# BGE-M3 + Qwen3-Embedding-4B on Kaggle

This notebook runs model inference only. It does **not** connect to Qdrant.

Pipeline:
1. Load 297 unique context chunks and 100 distinct QA questions.
2. Encode context chunks and questions with `BAAI/bge-m3` into dense and sparse vectors.
3. Encode context chunks and questions with `Qwen/Qwen3-Embedding-4B` into dense vectors.
4. Log model loading, document encoding, query encoding, and total elapsed time.
5. Save Qdrant-ready artifacts under `/kaggle/working/virhe4qa_embeddings`.

Model dimensions:

- BGE-M3 dense: 1,024 dimensions. Its sparse output is variable-length: token IDs plus lexical weights.
- Qwen3-Embedding-4B: dense only, up to 2,560 dimensions, with configurable Matryoshka dimensions. This notebook keeps the full 2,560 dimensions by default.
- Qwen3-Embedding-4B is open-sourced under Apache 2.0 and supports 100+ languages.

Attach the prepared `ViRHE4QA_v2_all_contexts_297` folder as a Kaggle Dataset before running the notebook.


In [ ]:
# Kaggle setup. The first run downloads Python packages and model weights.
!pip install -q -U 'FlagEmbedding>=1.3,<2' 'sentence-transformers>=2.7' accelerate pandas pyarrow

In [ ]:
from pathlib import Path
import gc
import glob
import hashlib
import json
import os
import time
import unicodedata

import numpy as np
import pandas as pd
import torch

SEED = 42
BGE_MODEL_NAME = 'BAAI/bge-m3'
QWEN_MODEL_NAME = 'Qwen/Qwen3-Embedding-4B'
BGE_BATCH_SIZE = 8
QWEN_BATCH_SIZE = 1  # 4B model: keep this small on a Kaggle T4
QWEN_OUTPUT_DIM = None  # None = full 2,560 dimensions; set 1024 for MRL truncation
MAX_LENGTH_BGE = 8192
MAX_LENGTH_QWEN = 8192
QWEN_QUERY_TASK = 'Given a user question, retrieve the most relevant Vietnamese passage.'
OUTPUT_DIR = Path('/kaggle/working/virhe4qa_embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timing_records = []

def timed_call(model_name, phase, fn, item_count=None):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    value = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    record = {
        'model': model_name,
        'phase': phase,
        'seconds': round(elapsed, 4),
        'items': item_count,
        'items_per_second': round(item_count / elapsed, 4) if item_count and elapsed else None,
    }
    timing_records.append(record)
    print(f"[timing] {model_name:<24} {phase:<16} {elapsed:>10.3f}s" + (f" | {item_count} items | {record['items_per_second']:.2f}/s" if item_count else ''))
    return value

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Locate the attached Kaggle Dataset.
chunk_candidates = []
query_candidates = []
for root in ('/kaggle/input', '/kaggle/working'):
    if os.path.exists(root):
        chunk_candidates.extend(glob.glob(root + '/**/ViRHE4QA_v2_all_contexts.csv', recursive=True))
        chunk_candidates.extend(glob.glob(root + '/**/ViRHE4QA_v2.csv', recursive=True))
        query_candidates.extend(glob.glob(root + '/**/queries_100.csv', recursive=True))
        query_candidates.extend(glob.glob(root + '/**/ViRHE4QA_v2_sample_100.csv', recursive=True))

chunk_candidates = sorted(set(chunk_candidates))
query_candidates = sorted(set(query_candidates))
if not chunk_candidates:
    raise FileNotFoundError('Attach the prepared chunks Kaggle Dataset first.')
if not query_candidates:
    raise FileNotFoundError('Attach the 100-question CSV Kaggle Dataset first.')

preferred_chunks = [p for p in chunk_candidates if p.endswith('ViRHE4QA_v2_all_contexts.csv')]
preferred_queries = [p for p in query_candidates if p.endswith('queries_100.csv')]
CHUNK_CSV = Path((preferred_chunks or chunk_candidates)[0])
QUERY_CSV = Path((preferred_queries or query_candidates)[0])
print('Chunks:', CHUNK_CSV)
print('Queries:', QUERY_CSV)


In [ ]:
# Build deterministic manifests: 297 context chunks and 100 query questions.
def normalize_context(value: str) -> str:
    value = (value or '').replace('\r\n', '\n').replace('\r', '\n').strip()
    return unicodedata.normalize('NFC', value)

def stable_document_id(text: str) -> str:
    digest = hashlib.sha256(text.encode('utf-8')).hexdigest()[:20]
    return f'virhe4qa_{digest}'

chunk_source_df = pd.read_csv(CHUNK_CSV, dtype=str, keep_default_na=False)
required_chunk_columns = {'context'}
missing = required_chunk_columns - set(chunk_source_df.columns)
if missing:
    raise ValueError(f'Missing chunk columns: {sorted(missing)}')
chunk_source_df['context_norm'] = chunk_source_df['context'].map(normalize_context)
chunk_source_df = chunk_source_df[chunk_source_df['context_norm'].ne('')].copy()
chunk_source_df['_index_num'] = pd.to_numeric(chunk_source_df.get('index', ''), errors='coerce').fillna(-1)
chunk_source_df = chunk_source_df.sort_values(['context_norm', '_index_num', 'split', 'question'])
chunks = chunk_source_df.drop_duplicates('context_norm', keep='first').sort_values('context_norm').reset_index(drop=True)
chunks['chunk_id'] = [f'chunk_{i:04d}' for i in range(1, len(chunks) + 1)]
chunks['document_id'] = chunks['context_norm'].map(stable_document_id)
chunks['text'] = chunks['context_norm']
chunks['point_key'] = chunks['document_id'] + ':' + chunks['chunk_id']
manifest = chunks[['chunk_id', 'document_id', 'point_key', 'split', 'index', 'label', 'text']].copy()
manifest.to_csv(OUTPUT_DIR / 'chunks_manifest.csv', index=False, encoding='utf-8')

query_source_df = pd.read_csv(QUERY_CSV, dtype=str, keep_default_na=False)
if 'question' not in query_source_df.columns or 'context' not in query_source_df.columns:
    raise ValueError('Query CSV must contain question and context columns')
query_source_df['context_norm'] = query_source_df['context'].map(normalize_context)
query_source_df['question_norm'] = query_source_df['question'].map(lambda value: (value or '').strip())
query_source_df = query_source_df[query_source_df['question_norm'].ne('')].copy()
query_source_df = query_source_df.drop_duplicates('question_norm').sort_values(['question_norm', 'context_norm']).reset_index(drop=True)
query_source_df = query_source_df.head(100).copy()
context_to_document = dict(zip(chunks['context_norm'], chunks['document_id']))
context_to_chunk = dict(zip(chunks['context_norm'], chunks['chunk_id']))
query_source_df['query_id'] = [f'query_{i:03d}' for i in range(1, len(query_source_df) + 1)]
query_source_df['gold_document_id'] = query_source_df['context_norm'].map(context_to_document)
query_source_df['gold_chunk_id'] = query_source_df['context_norm'].map(context_to_chunk)
if query_source_df['gold_document_id'].isna().any():
    raise ValueError('Every query context must exist in the 297-chunk manifest')
query_manifest = query_source_df[['query_id', 'gold_document_id', 'gold_chunk_id', 'split', 'index', 'label', 'question_norm', 'context_norm']].rename(columns={'question_norm': 'question', 'context_norm': 'gold_context'})
query_manifest.to_csv(OUTPUT_DIR / 'queries_manifest.csv', index=False, encoding='utf-8')

print('source chunk rows:', len(chunk_source_df))
print('unique context chunks:', len(manifest))
print('query questions:', len(query_manifest))
print('unique query texts:', query_manifest['question'].nunique())
display(manifest.head(3))
display(query_manifest.head(3))


## BGE-M3: dense + sparse

BGE-M3 exposes three retrieval representations. We export the dense representation and the learned lexical sparse representation; ColBERT multi-vector output is intentionally disabled for this run.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

bge_model = timed_call(
    'BGE-M3',
    'load model',
    lambda: BGEM3FlagModel(BGE_MODEL_NAME, use_fp16=torch.cuda.is_available()),
)

bge_document_outputs = timed_call(
    'BGE-M3',
    'encode documents',
    lambda: bge_model.encode(
        manifest['text'].tolist(),
        batch_size=BGE_BATCH_SIZE,
        max_length=MAX_LENGTH_BGE,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    ),
    item_count=len(manifest),
)

bge_query_outputs = timed_call(
    'BGE-M3',
    'encode queries',
    lambda: bge_model.encode(
        query_manifest['question'].tolist(),
        batch_size=BGE_BATCH_SIZE,
        max_length=MAX_LENGTH_BGE,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    ),
    item_count=len(query_manifest),
)

bge_dense = np.asarray(bge_document_outputs['dense_vecs'], dtype=np.float32)
bge_query_dense = np.asarray(bge_query_outputs['dense_vecs'], dtype=np.float32)

def sparse_rows(lexical_weights):
    rows = []
    for weights in lexical_weights:
        pairs = sorted((int(token_id), float(weight)) for token_id, weight in weights.items() if float(weight) != 0.0)
        rows.append({'indices': [token_id for token_id, _ in pairs], 'values': [weight for _, weight in pairs]})
    return rows

bge_sparse = sparse_rows(bge_document_outputs['lexical_weights'])
bge_query_sparse = sparse_rows(bge_query_outputs['lexical_weights'])
print('BGE document dense shape:', bge_dense.shape)
print('BGE query dense shape:', bge_query_dense.shape)
print('BGE document sparse rows:', len(bge_sparse))
print('BGE query sparse rows:', len(bge_query_sparse))


In [ ]:
# Save BGE-M3 document and query artifacts.
def write_sparse_jsonl(path, ids, sparse_rows_value):
    with path.open('w', encoding='utf-8') as stream:
        for item_id, sparse in zip(ids, sparse_rows_value):
            stream.write(json.dumps({
                'id': item_id,
                'indices': sparse['indices'],
                'values': sparse['values'],
            }, ensure_ascii=False) + '\n')

np.save(OUTPUT_DIR / 'bge_m3_dense.npy', bge_dense)
np.save(OUTPUT_DIR / 'bge_m3_query_dense.npy', bge_query_dense)
write_sparse_jsonl(OUTPUT_DIR / 'bge_m3_sparse.jsonl', manifest['chunk_id'].tolist(), bge_sparse)
write_sparse_jsonl(OUTPUT_DIR / 'bge_m3_query_sparse.jsonl', query_manifest['query_id'].tolist(), bge_query_sparse)

bge_metadata = {
    'model': BGE_MODEL_NAME,
    'num_documents': int(len(manifest)),
    'num_queries': int(len(query_manifest)),
    'dense_dimension': int(bge_dense.shape[1]),
    'sparse_format': 'jsonl indices + values; variable nnz per item',
    'source_chunks': CHUNK_CSV.name,
    'source_queries': QUERY_CSV.name,
}
(OUTPUT_DIR / 'bge_m3_metadata.json').write_text(json.dumps(bge_metadata, indent=2), encoding='utf-8')
print(bge_metadata)


In [ ]:
# Free BGE-M3 memory before loading the 4B Qwen embedding model.
del bge_model, bge_outputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('BGE-M3 released')

## Qwen3-Embedding-4B: dense only

Qwen3-Embedding-4B is an open-weight Apache 2.0 model. It produces dense embeddings only; it does not produce a BGE-style sparse lexical vector. The model supports instruction-aware query encoding, but documents are encoded without the query instruction.

In [ ]:
from sentence_transformers import SentenceTransformer

qwen_model = timed_call(
    'Qwen3-Embedding-4B',
    'load model',
    lambda: SentenceTransformer(
        QWEN_MODEL_NAME,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        model_kwargs={'torch_dtype': torch.float16} if torch.cuda.is_available() else {},
        tokenizer_kwargs={'padding_side': 'left'},
    ),
)
qwen_model.max_seq_length = MAX_LENGTH_QWEN

qwen_encode_kwargs = {
    'batch_size': QWEN_BATCH_SIZE,
    'normalize_embeddings': True,
    'show_progress_bar': True,
}
if QWEN_OUTPUT_DIM is not None:
    qwen_encode_kwargs['truncate_dim'] = QWEN_OUTPUT_DIM

qwen_dense = timed_call(
    'Qwen3-Embedding-4B',
    'encode documents',
    lambda: qwen_model.encode(manifest['text'].tolist(), **qwen_encode_kwargs),
    item_count=len(manifest),
)
qwen_query_dense = timed_call(
    'Qwen3-Embedding-4B',
    'encode queries',
    lambda: qwen_model.encode(query_manifest['question'].tolist(), prompt_name='query', **qwen_encode_kwargs),
    item_count=len(query_manifest),
)
qwen_dense = np.asarray(qwen_dense, dtype=np.float32)
qwen_query_dense = np.asarray(qwen_query_dense, dtype=np.float32)
print('Qwen document dense shape:', qwen_dense.shape)
print('Qwen query dense shape:', qwen_query_dense.shape)


In [ ]:
np.save(OUTPUT_DIR / 'qwen3_embedding_4b_dense.npy', qwen_dense)
np.save(OUTPUT_DIR / 'qwen3_embedding_4b_query_dense.npy', qwen_query_dense)

for model_name in ('BGE-M3', 'Qwen3-Embedding-4B'):
    total_seconds = sum(row['seconds'] for row in timing_records if row['model'] == model_name)
    timing_records.append({
        'model': model_name,
        'phase': 'total measured time',
        'seconds': round(total_seconds, 4),
        'items': None,
        'items_per_second': None,
    })
    print(f'[timing] {model_name:<24} {"total measured time":<16} {total_seconds:>10.3f}s')

timing_df = pd.DataFrame(timing_records)
timing_df.to_csv(OUTPUT_DIR / 'embedding_timings.csv', index=False)
(OUTPUT_DIR / 'embedding_timings.json').write_text(json.dumps(timing_records, indent=2), encoding='utf-8')

qwen_metadata = {
    'model': QWEN_MODEL_NAME,
    'num_documents': int(len(manifest)),
    'num_queries': int(len(query_manifest)),
    'dense_dimension': int(qwen_dense.shape[1]),
    'normalized': True,
    'query_prompt_name': 'query',
    'output_dimension_setting': QWEN_OUTPUT_DIM,
    'source_chunks': CHUNK_CSV.name,
    'source_queries': QUERY_CSV.name,
}
(OUTPUT_DIR / 'qwen3_embedding_4b_metadata.json').write_text(json.dumps(qwen_metadata, indent=2), encoding='utf-8')
print(qwen_metadata)
display(timing_df)


In [ ]:
# Final integrity checks and archive for download.
assert len(manifest) == 297, f'Expected 297 context chunks, got {len(manifest)}'
assert len(query_manifest) == 100, f'Expected 100 questions, got {len(query_manifest)}'
assert bge_dense.shape == (297, 1024), bge_dense.shape
assert bge_query_dense.shape == (100, 1024), bge_query_dense.shape
assert len(bge_sparse) == 297
assert len(bge_query_sparse) == 100
assert qwen_dense.shape[0] == 297
assert qwen_query_dense.shape[0] == 100
assert qwen_dense.shape[1] == qwen_query_dense.shape[1]

print('BGE document dense:', bge_dense.shape)
print('BGE query dense:', bge_query_dense.shape)
print('BGE document sparse:', len(bge_sparse), 'rows; mean nnz:', np.mean([len(row['indices']) for row in bge_sparse]))
print('BGE query sparse:', len(bge_query_sparse), 'rows; mean nnz:', np.mean([len(row['indices']) for row in bge_query_sparse]))
print('Qwen document dense:', qwen_dense.shape)
print('Qwen query dense:', qwen_query_dense.shape)
print('Artifacts:')
for path_item in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path_item.name}: {path_item.stat().st_size / 1024 / 1024:.2f} MB')

import shutil
archive_path = shutil.make_archive('/kaggle/working/virhe4qa_embeddings', 'zip', OUTPUT_DIR)
print('Download:', archive_path)


## Output mapping for the later Qdrant ingest

- `chunks_manifest.csv`: 297 stable chunks and source metadata.
- `queries_manifest.csv`: 100 questions with `gold_document_id` and `gold_chunk_id`.
- `bge_m3_dense.npy` / `bge_m3_query_dense.npy`: dense vectors, dimension 1,024.
- `bge_m3_sparse.jsonl` / `bge_m3_query_sparse.jsonl`: sparse vectors with token IDs and lexical weights.
- `qwen3_embedding_4b_dense.npy` / `qwen3_embedding_4b_query_dense.npy`: dense vectors, dimension 2,560 by default.
- `embedding_timings.csv` / `embedding_timings.json`: per-model load, document encoding, query encoding, and total timing.

BGE-M3 and Qwen vectors should be stored as separate named vectors or separate Qdrant collections because their dense dimensions differ. Qwen produces dense vectors only; the sparse branch in this notebook comes from BGE-M3.
